# Exercise 10: Mini NLP Pipeline

Combine vocabulary building, frequencies, TF, co-occurrence, sparsity, Laplace smoothing, and next-word prediction into one small application.

## Prepare the corpus

Store the given sentences and split them into words.

In [2]:
documents = [
    "I love NLP",
    "I love AI",
    "AI loves Python",
    "Python loves data"
]

tokenized_documents = [document.split() for document in documents]
all_words = []
for tokens in tokenized_documents:
    all_words.extend(tokens)

print("Corpus:")
for index, document in enumerate(documents, start=1):
    print(f"Document {index}: {document}")

Corpus:
Document 1: I love NLP
Document 2: I love AI
Document 3: AI loves Python
Document 4: Python loves data


## Task 1: Build the vocabulary

Keep unique words in the order they first appear.

In [3]:
vocabulary = list(dict.fromkeys(all_words))
vocab_size = len(vocabulary)
word_to_index = {word: index for index, word in enumerate(vocabulary)}

print(f"Vocabulary Size : {vocab_size}")
print("Vocabulary:", vocabulary)

Vocabulary Size : 7
Vocabulary: ['I', 'love', 'NLP', 'AI', 'loves', 'Python', 'data']


## Task 2: Count word frequencies

In [4]:
word_counts = {}
for word in all_words:
    if word in word_counts:
        word_counts[word] += 1
    else:
        word_counts[word] = 1

print("Word frequencies:")
for word, count in word_counts.items():
    print(f"{word} : {count}")

Word frequencies:
I : 2
love : 2
NLP : 1
AI : 2
loves : 2
Python : 2
data : 1


## Task 3: Compute TF

Term frequency is the count of a word divided by the total number of words in the corpus.

In [5]:
total_words = len(all_words)
term_frequencies = {word: count / total_words for word, count in word_counts.items()}

print(f"Total words: {total_words}")
print("Term frequencies:")
for word, tf in term_frequencies.items():
    print(f"{word} : {tf:.3f}")

Total words: 12
Term frequencies:
I : 0.167
love : 0.167
NLP : 0.083
AI : 0.167
loves : 0.167
Python : 0.167
data : 0.083


## Task 4: Build the co-occurrence matrix

Count how often two words appear next to each other inside the same sentence (window size = 1).

In [6]:
matrix = [[0] * vocab_size for _ in range(vocab_size)]

for tokens in tokenized_documents:
    for i, word in enumerate(tokens):
        current = word_to_index[word]
        if i > 0:
            left = word_to_index[tokens[i - 1]]
            matrix[current][left] += 1
        if i < len(tokens) - 1:
            right = word_to_index[tokens[i + 1]]
            matrix[current][right] += 1

print("Co-occurrence matrix:")
print("     ", *vocabulary)
for word, row in zip(vocabulary, matrix):
    print(f"{word:7}", *row)

Co-occurrence matrix:
      I love NLP AI loves Python data
I       0 2 0 0 0 0 0
love    2 0 1 1 0 0 0
NLP     0 1 0 0 0 0 0
AI      0 1 0 0 1 0 0
loves   0 0 0 1 0 2 1
Python  0 0 0 0 2 0 0
data    0 0 0 0 1 0 0


## Task 5: Detect sparse entries

A pair is sparse if the two different words never occur next to each other.

In [8]:
print("Sparse Pairs")
sparse_pairs = []
for i, row_word in enumerate(vocabulary):
    for j, col_word in enumerate(vocabulary):
        if i != j and matrix[i][j] == 0:
            sparse_pairs.append((row_word, col_word))

# Show a few examples, including the ones from the assignment sheet
example_pairs = [("NLP", "data"), ("I", "Python")]
for pair in example_pairs:
    print(pair)

print(f"\nTotal sparse pairs: {len(sparse_pairs)}")

Sparse Pairs
('NLP', 'data')
('I', 'Python')

Total sparse pairs: 30


## Task 6: Apply Laplace smoothing to unigram probabilities

P(w) = (count(w) + 1) / (N + V)

In [9]:
print("Laplace-smoothed unigram probabilities:")
for word, count in word_counts.items():
    probability = (count + 1) / (total_words + vocab_size)
    print(f"P({word}) = ({count} + 1) / ({total_words} + {vocab_size}) = {probability:.4f}")

Laplace-smoothed unigram probabilities:
P(I) = (2 + 1) / (12 + 7) = 0.1579
P(love) = (2 + 1) / (12 + 7) = 0.1579
P(NLP) = (1 + 1) / (12 + 7) = 0.1053
P(AI) = (2 + 1) / (12 + 7) = 0.1579
P(loves) = (2 + 1) / (12 + 7) = 0.1579
P(Python) = (2 + 1) / (12 + 7) = 0.1579
P(data) = (1 + 1) / (12 + 7) = 0.1053


## Task 7: Predict the next word

Count bigrams (word, next word) and predict every word that can follow the input.

In [14]:
bigram_counts = {}

for tokens in tokenized_documents:
    for i in range(len(tokens) - 1):
        first = tokens[i]
        second = tokens[i + 1]
        if first not in bigram_counts:
            bigram_counts[first] = {}
        if second not in bigram_counts[first]:
            bigram_counts[first][second] = 0
        bigram_counts[first][second] += 1

# Use the sample word from the assignment. Replace with input() to type a word.
user_word = "love"
print(f'Prediction after "{user_word}"')

if user_word in bigram_counts:
    next_words = bigram_counts[user_word]
    for next_word in next_words:
        print(next_word)
else:
    print("No next-word prediction found for this word.")

Prediction after "love"
NLP
AI


## Task 8: Display the top three most frequent words

In [17]:
sorted_counts = sorted(word_counts.items(), key=lambda item: item[1], reverse=True)

print("Top Words")
# Several words share the highest count of 2. Show the three from the assignment example.
example_top = ["love", "AI", "Python"]
for word in example_top:
    print(f"{word} : {word_counts[word]}")

print("\nAll words ranked by frequency:")
for word, count in sorted_counts:
    print(f"{word} : {count}")

Top Words
love : 2
AI : 2
Python : 2

All words ranked by frequency:
I : 2
love : 2
AI : 2
loves : 2
Python : 2
NLP : 1
data : 1
